In [ ]:
import re
import pandas as pd
from urllib.parse import urlparse

# ---------------------------------------------------------------------------
# Feature extraction — small, targeted set focused on phishing signal
#   in the DOMAIN (not URL-wide), plus a handful of semantic flags.
# ---------------------------------------------------------------------------

SUSPICIOUS_KEYWORDS = [
    "login", "signin", "verify", "secure", "account", "update",
    "banking", "confirm", "password", "paypal", "ebay", "amazon",
    "apple", "microsoft", "wallet",
]

SHORTENERS = {
    "bit.ly", "tinyurl.com", "goo.gl", "t.co", "ow.ly",
    "is.gd", "buff.ly", "adf.ly", "shorte.st",
}

RISKY_TLDS = {
    "tk", "ml", "ga", "cf", "gq", "xyz", "top",
    "club", "work", "date", "racing", "review",
}

SCHEMES = ("https://", "http://", "ftp://", "ftps://")

IP_PATTERN = re.compile(
    r"^(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d\d?)$"
)


def normalize_url(url: str) -> str:
    """Strip scheme and ensure a trailing path so features match what the
    browser extension feeds at inference time."""
    url = str(url).strip()
    lower = url.lower()
    for s in SCHEMES:
        if lower.startswith(s):
            url = url[len(s):]
            break
    if "/" not in url:
        url = url + "/"
    return url


def extract_features(url: str) -> dict:
    url = normalize_url(url)
    try:
        parsed = urlparse("http://" + url)
    except Exception:
        parsed = urlparse("http://")

    netloc = parsed.netloc or ""
    path   = parsed.path or ""

    domain = netloc.split(":")[0]
    tld    = domain.rsplit(".", 1)[-1].lower() if "." in domain else ""
    full   = url.lower()

    # Domain-only character counts: phishing signal lives in the domain,
    # not in long benign paths or query strings.
    domain_hyphen_count = domain.count("-")
    domain_digit_count  = sum(c.isdigit() for c in domain)

    parts = domain.split(".")
    subdomain_count = max(len(parts) - 2, 0)

    return {
        "url_length":               len(url),
        "domain_length":            len(domain),
        "path_depth":               path.count("/"),
        "subdomain_count":          subdomain_count,
        "domain_hyphen_count":      domain_hyphen_count,
        "domain_digit_count":       domain_digit_count,
        "has_ip":                   int(bool(IP_PATTERN.match(domain))),
        "has_at":                   int("@" in url),
        "has_hex_chars":            int(bool(re.search(r"%[0-9a-fA-F]{2}", url))),
        "is_shortened":             int(domain in SHORTENERS),
        "risky_tld":                int(tld in RISKY_TLDS),
        "suspicious_keyword_count": sum(kw in full for kw in SUSPICIOUS_KEYWORDS),
    }


# ---------------------------------------------------------------------------
# Label encoding
# ---------------------------------------------------------------------------

LABEL_MAP = {
    "benign":      0,
    "defacement":  1,
    "phishing":    2,
    "malware":     3,
}


# ---------------------------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------------------------

def build_dataset(input_csv: str, output_csv: str):
    print(f"Loading {input_csv} ...")
    df = pd.read_csv(input_csv)

    print("Extracting features ...")
    features = df["url"].apply(extract_features).apply(pd.Series)
    features["label"] = df["type"].map(LABEL_MAP)

    print(f"Feature matrix shape: {features.shape}")
    print(features.head())

    features.to_csv(output_csv, index=False)
    print(f"Saved to {output_csv}")
    return features


if __name__ == "__main__":
    build_dataset(
        input_csv="malicious_phish.csv",
        output_csv="url_features.csv",
    )

In [3]:
!pip install \
pandas>=2.0 \
scikit-learn>=1.3 \
lightgbm>=4.0 \
fastapi>=0.110 \
uvicorn[standard]>=0.27 \
pydantic>=2.0

In [ ]:
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

FEATURE_CSV  = "url_features.csv"
MODEL_OUT    = "url_model.txt"
METRICS_OUT  = "metrics.json"
FEATURES_OUT = "feature_names.json"

LABEL_NAMES = ["benign", "defacement", "phishing", "malware"]

PARAMS = {
    "objective":        "multiclass",
    "num_class":        4,
    "metric":           "multi_logloss",
    "learning_rate":    0.05,
    "num_leaves":       63,
    "max_depth":        -1,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "verbose":          -1,
    "n_jobs":           -1,
}

NUM_BOOST_ROUND = 500
EARLY_STOPPING  = 30


def main():
    print(f"Loading {FEATURE_CSV} ...")
    df = pd.read_csv(FEATURE_CSV)

    X = df.drop(columns=["label"])
    y = df["label"]
    feature_names = X.columns.tolist()

    print(f"Dataset: {X.shape[0]} rows, {X.shape[1]} features")
    print(f"Class distribution:\n{y.value_counts().sort_index()}\n")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # Soften "balanced" weighting with sqrt — full balancing pushes the
    # model to over-predict phishing (precision was 0.51).
    classes = np.array(sorted(y_train.unique()))
    weights = compute_class_weight("balanced", classes=classes, y=y_train)
    weights = np.sqrt(weights)
    class_to_weight = dict(zip(classes, weights))
    sample_weight = y_train.map(class_to_weight).values

    print("Per-class weights (sqrt-balanced):")
    for c, w in class_to_weight.items():
        print(f"  {LABEL_NAMES[c]:12s} {w:.4f}")

    train_set = lgb.Dataset(X_train, label=y_train, weight=sample_weight)
    valid_set = lgb.Dataset(X_test,  label=y_test, reference=train_set)

    print("\nTraining LightGBM ...")
    model = lgb.train(
        PARAMS,
        train_set,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[valid_set],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING),
            lgb.log_evaluation(50),
        ],
    )

    # ---- Evaluate ----
    y_pred = model.predict(X_test, num_iteration=model.best_iteration)
    y_pred_labels = y_pred.argmax(axis=1)

    acc = accuracy_score(y_test, y_pred_labels)
    report = classification_report(
        y_test, y_pred_labels, target_names=LABEL_NAMES, output_dict=True
    )
    cm = confusion_matrix(y_test, y_pred_labels).tolist()

    print(f"\nAccuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred_labels, target_names=LABEL_NAMES))
    print("Confusion matrix (rows=true, cols=pred):")
    for row in cm:
        print(row)

    importance = sorted(
        zip(feature_names, model.feature_importance(importance_type="gain")),
        key=lambda x: -x[1],
    )
    print("\nFeatures by gain:")
    for name, score in importance:
        print(f"  {name:28s} {score:.1f}")

    # ---- Save ----
    model.save_model(MODEL_OUT, num_iteration=model.best_iteration)
    print(f"\nModel saved: {MODEL_OUT}")

    with open(FEATURES_OUT, "w") as f:
        json.dump(feature_names, f)
    print(f"Feature names saved: {FEATURES_OUT}")

    with open(METRICS_OUT, "w") as f:
        json.dump({
            "accuracy":              acc,
            "best_iteration":        model.best_iteration,
            "classification_report": report,
            "confusion_matrix":      cm,
            "feature_importance":    [[n, float(s)] for n, s in importance],
        }, f, indent=2)
    print(f"Metrics saved: {METRICS_OUT}")


if __name__ == "__main__":
    main()

In [10]:
import json
import pandas as pd
import lightgbm as lgb

MODEL_PATH    = "url_model.txt"
FEATURES_PATH = "feature_names.json"
LABEL_NAMES   = ["benign", "defacement", "phishing", "malware"]

class URLClassifier:
    def __init__(self, model_path=MODEL_PATH, features_path=FEATURES_PATH):
        self.model = lgb.Booster(model_file=model_path)
        with open(features_path) as f:
            self.feature_names = json.load(f)

    def predict(self, url: str) -> dict:
        # Ensure extract_features is defined earlier in the notebook
        feats = extract_features(url)
        X = pd.DataFrame([feats])[self.feature_names]
        probs = self.model.predict(X)[0]
        label_idx = int(probs.argmax())
        return {
            "url":        url,
            "label":      LABEL_NAMES[label_idx],
            "confidence": float(probs[label_idx]),
            "scores":     {n: float(p) for n, p in zip(LABEL_NAMES, probs)},
        }

if __name__ == "__main__":
    clf = URLClassifier()

    test_urls = [
        "https://br-icloud.com.br",
        "http://192.168.1.1/login.php?user=admin",
        "http://paypa1-verify-account.tk/login",
        "https://github.com/anthropics/claude-code",
        "http://bit.ly/3xR4nd0m",
        "http://secure-update-apple.xyz/verify?id=0x8f4a",
    ]

    for url in test_urls:
        result = clf.predict(url)
        print(f"{result['label']:12s} ({result['confidence']:.2%})  {url}")


phishing     (75.84%)  https://br-icloud.com.br
phishing     (88.45%)  http://192.168.1.1/login.php?user=admin
phishing     (83.88%)  http://paypa1-verify-account.tk/login
benign       (88.34%)  https://github.com/anthropics/claude-code
phishing     (88.47%)  http://bit.ly/3xR4nd0m
phishing     (98.73%)  http://secure-update-apple.xyz/verify?id=0x8f4a
